# What is the surface wind speed?

ERA5 hourly data on single levels from 1940 to present reports 10 m wind speeds. Their products include a single run on a high resolution grid (variables named `*_prod` below) and an ensemble of 10 members on a coarser reslution grid. The low resolution ensemble reports members (not shown), mean, and the spread (standard deviation of the 10 members). The spread is a measure of relative uncertainty and not absolute uncertainty. See [this](https://confluence.ecmwf.int/display/CKB/ERA5%3A+uncertainty+estimation) for more info.

In [1]:
# conda env spec
from pathlib import Path
import xarray as xr
import numpy as np

PROJECT_ROOT = Path().resolve().parent.parent.parent

In [2]:
RAW_DATA = PROJECT_ROOT / 'data' / 'raw'
era_prod = xr.load_dataset(RAW_DATA / 'era5' / 'ERA5_surface_CAMP2Ex_RF17_wide.nc')
era_mean = xr.load_dataset(RAW_DATA / 'era5' / 'ERA5_surface_CAMP2Ex_RF17_wide_ensemble_mean.nc')
era_spread = xr.load_dataset(RAW_DATA / 'era5' / 'ERA5_surface_CAMP2Ex_RF17_wide_ensemble_spread.nc')

era_prod['s10'] = (era_prod.u10**2 + era_prod.v10**2)**0.5
era_mean['s10'] = (era_mean.u10**2 + era_mean.v10**2)**0.5

target_lat = 19.266
target_lon = 119.724
target_time = np.datetime64('2019-10-02T02:40:00')

In [3]:
interp_kwargs = {
    'valid_time': target_time,
    'latitude': target_lat,
    'longitude': target_lon,
    'method': 'linear'
}
e_prod = era_prod.interp(**interp_kwargs)
mean = era_mean.interp(**interp_kwargs)
spread = era_spread.interp(**interp_kwargs)

In [4]:
print(((mean.u10**2 + mean.v10**2)**0.5).data)
print((((mean.u10+spread.u10)**2 + mean.v10**2)**0.5).data)
print((((mean.u10-spread.u10)**2 + mean.v10**2)**0.5).data)
print((((mean.u10+spread.u10)**2 + (mean.v10+spread.v10)**2)**0.5).data)
print((((mean.u10-spread.u10)**2 + (mean.v10-spread.v10)**2)**0.5).data)
print((((mean.u10+spread.u10)**2 + (mean.v10-spread.v10)**2)**0.5).data)
print((((mean.u10-spread.u10)**2 + (mean.v10+spread.v10)**2)**0.5).data)
print((((mean.u10)**2 + (mean.v10+spread.v10)**2)**0.5).data)
print((((mean.u10)**2 + (mean.v10-spread.v10)**2)**0.5).data)

0.9558172939484801
1.1966773913041906
0.7182907831616928
1.1826808984093486
0.8696681638759536
1.2932199522675705
0.6947222306640108
0.9382345271917552
1.0742289167097159


In [5]:
smin = 0.6947222306640108 # interp method: linear
smax = 1.2932199522675705 # interp method: linear
# smin = 1.0797491 # interp method: nearest
# smax = 1.6122911 # interp method: nearest
print((smin+smax)/2)
print(mean.s10.data)

0.9939710914657907
1.0156876352945938


In [6]:
print(e_prod.s10.data)

0.7080524135665888
